# Part 17 — A new venue and an order-splitting placebo (Paper 2)
Every earlier test used Binance, Bybit or US stocks. This notebook runs a **pre-registered** test on **Coinbase Exchange spot**, a US-regulated venue the program has never used, with public trade data only (no account and no API keys).

**Two primaries, tested in a fixed sequence:**
1. In thin Coinbase USD pairs, the depth-free amplification margin `C3_flow` forecasts the next hour's adverse excursion beyond a volatility and price-impact baseline.
2. Only if 1 passes: the real statistic beats an **order-splitting placebo**, the same statistic computed after shifting the flow series by 180 minutes inside its own six-hour window. The placebo keeps the flow's size distribution and persistence (order splitting) but breaks its minute-by-minute alignment with prices, which is what the model says matters.

**Rules:**
1. Run the cells one at a time, in order.
2. **C3 downloads `P17_new_venue.json`: e-mail it to yourself before running C4.**
3. C4 is the long one (about 2–4 hours at Coinbase's public rate limit). It caches every pair-day in Drive, so if Colab disconnects, re-run C0, C1, C2, C3 and then C4: it resumes where it stopped.
4. Run C6 **once**. Then run C7 and send me `p17_results.zip`.

In [ ]:
# ============================================================================
# CELL C0 — SETUP (self-contained; run first)
# Part 17 (Paper 2): A NEW VENUE AND AN ORDER-SPLITTING PLACEBO.
# Coinbase Exchange spot (a US-regulated venue never used in Parts 1-16), public trade data only (no account, no keys).
# Two pre-registered primaries: (1) the depth-free amplification margin forecasts the next hour's adverse excursion beyond a
# volatility and impact baseline in thin Coinbase pairs; (2) the real statistic beats an ORDER-SPLITTING PLACEBO computed from
# the same flow with its minute alignment to prices broken (same flow distribution and persistence).
# ============================================================================
import os, sys, json, time, math, io, zipfile, hashlib, subprocess, warnings, threading
import urllib.request, urllib.error, urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
CDATA = os.path.join(BASE, "data_coinbase")
for _d in (RESULTS, PREREG, CDATA, os.path.join(CDATA, "minute"), os.path.join(CDATA, "bounds")):
    os.makedirs(_d, exist_ok=True)

SEED = 20261117
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    return out

# ---- Calendar (fixed here, before anything is downloaded) -------------------------------------
TRAIN_DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range("2024-01-01", "2024-12-31") if d.day in (10, 24)]
TEST_DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range("2025-01-01", "2026-09-30") if d.day in (6, 20)]
TRAIN_END = "2024-12-31"
HOURS_EVAL = list(range(6, 24))          # forecast hours (UTC); features from [t-360, t), outcome over (t, t+60]
PLACEBO_SHIFT = 180                      # minutes; circular shift of flow inside the 360-minute window
print(f"BASE = {BASE} | {len(TRAIN_DAYS)} training and {len(TEST_DAYS)} test days")

# ---- HTTP with a shared rate limit (Coinbase public endpoints: 10 requests/s per IP; we use 7) ----
API = "https://api.exchange.coinbase.com"
_lock = threading.Lock(); _last = [0.0]; RATE = 7.0
def _throttle():
    with _lock:
        wait = _last[0] + 1.0 / RATE - time.time()
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.time()
N_REQ = [0]
def api(path, params=None, tries=8):
    url = API + path + ("?" + urllib.parse.urlencode(params) if params else "")
    last = None
    for i in range(tries):
        _throttle()
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "research-replication/1.0", "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=60) as r:
                N_REQ[0] += 1
                return json.loads(r.read().decode())
        except urllib.error.HTTPError as e:
            last = e
            if e.code == 404:
                return None
            time.sleep(min(30, 1.5 * 2 ** i) if e.code == 429 else 1.5 * (i + 1))
        except Exception as e:
            last = e; time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"API error {url}: {last}")


In [ ]:
# ============================================================================
# CELL C1 — FUNCTIONS (Coinbase trade loader, minute bars, depth-free features, evaluation). Defines only; nothing runs.
# ============================================================================
STABLE = {"USDT", "USDC", "DAI", "PYUSD", "EURC", "GUSD", "PAX", "PAXG", "BUSD", "UST", "TUSD", "USDS", "RLUSD", "FDUSD",
          "WBTC", "CBETH", "MSOL", "LSETH", "WETH", "CBBTC", "STETH", "WAXL"}

def _t(s):
    return pd.Timestamp(s).tz_convert("UTC") if pd.Timestamp(s).tzinfo else pd.Timestamp(s, tz="UTC")

OLDER = {"param": None}
def check_pagination(pid):
    """Find which cursor parameter returns OLDER trades (ids below the cursor). Recorded in C2."""
    p1 = api(f"/products/{pid}/trades", {"limit": 100})
    c = min(t["trade_id"] for t in p1)
    for prm in ("after", "before"):
        p2 = api(f"/products/{pid}/trades", {"limit": 100, prm: c})
        if p2 and max(t["trade_id"] for t in p2) < c:
            OLDER["param"] = prm
            return prm
    raise RuntimeError("could not determine the pagination direction")

def older_page(pid, cursor, limit=1000):
    """Trades with trade_id < cursor, newest first."""
    return api(f"/products/{pid}/trades", {"limit": limit, OLDER["param"]: int(cursor)}) or []

def trade_at(pid, tid):
    pg = older_page(pid, tid + 1, limit=1)
    return (pg[0]["trade_id"], _t(pg[0]["time"])) if pg else (None, None)

_BLOCK = threading.Lock(); _BMEM = {}
def _bounds(pid):
    if pid not in _BMEM:
        fp = os.path.join(CDATA, "bounds", f"{pid}.json")
        try:
            _BMEM[pid] = json.load(open(fp)) if os.path.exists(fp) else {}
        except Exception:
            _BMEM[pid] = {}
    return _BMEM[pid]

def cursor_at(pid, T):
    """A trade id whose time is >= T and that is within ~1,000 ids of the first such trade (cached)."""
    key = T.strftime("%Y-%m-%dT%H:%M:%S")
    with _BLOCK:
        cache = _bounds(pid)
        if key in cache:
            return cache[key]
    top = api(f"/products/{pid}/trades", {"limit": 1})
    hi_id, hi_t = top[0]["trade_id"], _t(top[0]["time"])
    if hi_t < T:
        return None
    lo_id, lo_t = trade_at(pid, 1)
    if lo_id is None:                                   # ids do not start at 1: find the first one
        lo_id, lo_t = 0, pd.Timestamp("2010-01-01", tz="UTC")
    if lo_t >= T:
        res = lo_id
    else:
        it = 0
        while hi_id - lo_id > 1000 and it < 60:
            it += 1
            if it % 3 == 0:
                g = (lo_id + hi_id) // 2
            else:
                fr = (T - lo_t) / (hi_t - lo_t) if hi_t > lo_t else 0.5
                g = int(lo_id + min(max(fr, 0.02), 0.98) * (hi_id - lo_id))
            g = min(max(g, lo_id + 1), hi_id - 1)
            gid, gt = trade_at(pid, g)
            if gid is None or gid <= lo_id:
                lo_id = g                                   # gap in ids: treat as below
                continue
            if gt < T:
                lo_id, lo_t = gid, gt
            else:
                hi_id, hi_t = gid, gt
        res = hi_id
    with _BLOCK:
        cache = _bounds(pid); cache[key] = int(res)
        fp = os.path.join(CDATA, "bounds", f"{pid}.json"); tmp = fp + f".tmp{threading.get_ident()}"
        json.dump(cache, open(tmp, "w")); os.replace(tmp, fp)             # atomic: safe with several threads
    return int(res)

MAX_PAGES = 400
def build_day(pid, date):
    """Minute bars for one UTC day: close (last trade), signed taker flow ($M; + = taker buy), volume ($M), trades."""
    t0 = _t(f"{date} 00:00:00"); t1 = t0 + pd.Timedelta("1D")
    cur = cursor_at(pid, t1)
    if cur is None:
        return None, {"symbol": pid, "date": date, "reason": "day not complete"}
    rows, pages = [], 0
    while pages < MAX_PAGES:
        pg = older_page(pid, cur); pages += 1
        if not pg:
            break
        rows += pg; cur = min(t["trade_id"] for t in pg)
        if _t(pg[-1]["time"]) < t0:                       # pages are newest first; the last trade is the oldest
            break
    if pages >= MAX_PAGES:
        return None, {"symbol": pid, "date": date, "reason": "too many trades", "pages": pages}
    if not rows:
        return None, {"symbol": pid, "date": date, "reason": "no trades"}
    T = pd.DataFrame(rows)
    T["ts"] = pd.to_datetime(T["time"], utc=True, format="ISO8601")
    T = T[(T["ts"] >= t0) & (T["ts"] < t1)].drop_duplicates("trade_id").sort_values("trade_id")
    if T.empty:
        return None, {"symbol": pid, "date": date, "reason": "no trades in day"}
    px, sz = T["price"].astype(float).to_numpy(), T["size"].astype(float).to_numpy()
    notional = px * sz / 1e6
    sign = np.where(T["side"].to_numpy() == "sell", 1.0, -1.0)        # maker sell = taker buy (up-tick)
    m = ((T["ts"] - t0).dt.total_seconds() // 60).astype(int).clip(0, 1439).to_numpy()
    idx = np.arange(1440)
    g = pd.DataFrame({"m": m, "px": px, "f": sign * notional, "v": notional}).groupby("m")
    out = pd.DataFrame(index=idx)
    out["close"] = g["px"].last().reindex(idx).ffill().bfill()
    out["flow"] = g["f"].sum().reindex(idx).fillna(0.0)
    out["vol"] = g["v"].sum().reindex(idx).fillna(0.0)
    out["ntr"] = g["px"].size().reindex(idx).fillna(0).astype(int)
    cov = float((out["ntr"] > 0).mean())
    st = {"symbol": pid, "date": date, "trades": int(len(T)), "pages": pages, "minute_coverage": cov,
          "day_ok": bool(cov >= MIN_COVERAGE), "first_id": int(T["trade_id"].iloc[0]), "last_id": int(T["trade_id"].iloc[-1]),
          "sha256": hashlib.sha256(pd.util.hash_pandas_object(T[["trade_id", "price", "size", "side"]], index=False).values.tobytes()).hexdigest()}
    out["symbol"], out["date"] = pid, date
    return out, st

MIN_COVERAGE = 0.60
def get_day(pid, date):
    d_ = os.path.join(CDATA, "minute", pid); os.makedirs(d_, exist_ok=True)
    pq, js = os.path.join(d_, f"{date}.parquet"), os.path.join(d_, f"{date}.json")
    if os.path.exists(js):
        st = json.load(open(js))
        return (pd.read_parquet(pq) if os.path.exists(pq) else None), st
    df, st = build_day(pid, date)
    if df is not None:
        df.to_parquet(pq)
    json.dump(_clean(st), open(js, "w"))
    return df, st

# ---- estimation helpers (same as Parts 9, 14 and 16) ------------------------------------------
def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    return np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0][1:]

def impulse_2s(a, kappa, eta, q, L=60):
    x = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        x[t] = (a * x[t - 1] if t else 0) + kappa * ((1.0 if t == 0 else 0.0) + f[t])
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x

def c3_flow(dpw, uw, xw):
    """Depth-free amplification margin and summed response from one window (prices and signed flow only)."""
    bp0 = ols(dpw[1:], np.column_stack([uw[1:], xw[:-1]]))
    bm0 = ols(uw[1:], np.column_stack([uw[:-1], xw[:-1]]))
    if bp0 is None or bm0 is None or abs(1 - bm0[0]) < 1e-6:
        return None
    k0, a0, e0 = bp0[0], 1 + bp0[1], 1 - bm0[0]; q0 = bm0[1] / e0
    imp = impulse_2s(a0, k0, e0, q0)
    return {"C3": k0 ** 2 * e0 * q0 - (1 - a0) * k0, "kappa": k0, "a": a0,
            "S60": float(np.sum(np.abs(imp))) if np.all(np.isfinite(imp)) else np.nan}

def vpin_window(vol, flow, nb=50):
    v = np.nan_to_num(vol); f = np.nan_to_num(flow); tot = v.sum()
    if tot <= 0:
        return np.nan
    buy = (v + f) / 2; cum = np.cumsum(v); edges = np.linspace(0, tot, nb + 1)
    k = np.clip(np.searchsorted(edges, cum, side="left") - 1, 0, nb - 1)
    imb = np.zeros(nb); np.add.at(imb, k, buy - (v - buy))
    return float(np.sum(np.abs(imb)) / tot)

HOURS = list(range(7, 24))
def cb_blocks(df, s, d, flow_scale):
    p = np.log(df["close"].to_numpy()) * 1e4; u = df["flow"].to_numpy() / flow_scale
    vol, flow = df["vol"].to_numpy(), df["flow"].to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy(); dp = np.diff(p, prepend=np.nan)
    ew = pd.Series(np.nan_to_num(dp) ** 2).ewm(alpha=0.06, adjust=False).mean().to_numpy()
    rows = []
    for hr in HOURS_EVAL:
        t = hr * 60; w, h, f = slice(t - 360, t), slice(t - 60, t), slice(t, min(t + 61, 1440))
        row = {"symbol": s, "date": d, "hour": hr}
        real = c3_flow(dp[w], u[w], x[w])
        plac = c3_flow(dp[w], np.roll(u[w], PLACEBO_SHIFT), x[w])          # order-splitting placebo
        if real is not None:
            row.update({"C3_flow": real["C3"], "S60": real["S60"], "kappa_hat": real["kappa"]})
        if plac is not None:
            row.update({"C3_placebo": plac["C3"], "S60_placebo": plac["S60"]})
        uu, yy = u[w][1:], dp[w][1:]; okk = np.isfinite(uu) & np.isfinite(yy)
        v60 = np.nansum(vol[h])
        row.update({"rv_60": np.nanstd(np.diff(p[h])), "rv_360": np.nanstd(np.diff(p[w])), "abs_ret_60": abs(p[t - 1] - p[t - 60]),
                    "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12), "log_vol_60": np.log1p(v60),
                    "kyle_lam": float(np.polyfit(uu[okk], yy[okk], 1)[0]) if okk.sum() > 60 and np.std(uu[okk]) > 0 else np.nan,
                    "amihud_60": np.log(abs(p[t - 1] - p[t - 60]) / v60 + 1e-9) if v60 > 0 else np.nan,
                    "rv_15": np.nanstd(dp[t - 15:t]), "rv_day": np.nanstd(dp[1:t]), "ewma_vol": float(np.sqrt(ew[t - 1])),
                    "mae_lag60": float(np.nanmax(np.abs(p[t - 60:t] - p[t - 61]))),
                    "vpin": vpin_window(vol[w], flow[w]), "trades_6h": int(df["ntr"].to_numpy()[w].sum()),
                    "MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1])))})
        for hh in HOURS:
            row[f"hr_{hh}"] = float(hr == hh)
        rows.append(row)
    return rows

CB_CTRL = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "kappa_hat", "kappa_sq", "kyle_lam", "amihud_60",
           "rv_15", "rv_day", "ewma_vol", "mae_lag60"] + [f"hr_{h}" for h in HOURS]

def winsor_train(D, cols):
    D = D.copy()
    for c in cols:
        lo, hi = D.loc[D["split"] == "train", c].quantile([0.01, 0.99])
        D[c] = D[c].clip(lo, hi)
    return D

def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def cw_test(y, fb, ff, lags=6):
    adj = (y - fb) ** 2 - ((y - ff) ** 2 - (fb - ff) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))

def design(D, cols, syms):
    X = [np.ones(len(D))] + [D[c].to_numpy() for c in cols] + [(D["symbol"] == s).to_numpy(float) for s in syms[1:]]
    return np.column_stack(X)

def _fit_predict(A, Bt, outcome, cols, syms):
    b = np.linalg.lstsq(design(A, cols, syms), A[outcome].to_numpy(), rcond=None)[0]
    return design(Bt, cols, syms) @ b

def evaluate_set(sub, outcome, cand_sets, ctrl, B=2000, level=0.95, seed=88):
    """Out-of-sample evaluation of several candidate sets against one baseline on the SAME rows and the SAME bootstrap
    draws (so differences between candidates are paired). Candidates winsorized at training 1%/99%; symbol fixed effects."""
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    allc = sorted({c for cs in cand_sets.values() for c in cs})
    for c in allc:
        lo, hi = A0[c].quantile([0.01, 0.99]); A0[c + "_w"] = A0[c].clip(lo, hi); B0[c + "_w"] = B0[c].clip(lo, hi)
    need = ctrl + [c + "_w" for c in allc] + [outcome]
    A = A0.dropna(subset=need); Bt = B0.dropna(subset=need)
    Bt = Bt[Bt["symbol"].isin(set(A["symbol"]))].sort_values(["date", "symbol", "hour"])
    syms = sorted(A["symbol"].unique()); y = Bt[outcome].to_numpy()
    fb = _fit_predict(A, Bt, outcome, ctrl, syms)
    ff = {k: _fit_predict(A, Bt, outcome, ctrl + [c + "_w" for c in cs], syms) for k, cs in cand_sets.items()}
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2); tail = (y > A[outcome].quantile(0.9)).astype(int)
    out = {"n_train": len(A), "n_test": len(Bt), "n_symbols": len(syms), "n_test_dates": int(Bt["date"].nunique()),
           "r2_base": float(1 - np.sum((y - fb) ** 2) / sst), "auc_base": auc_(tail, fb), "cands": {}}
    Rb = rng(seed); dd = Bt["date"].to_numpy(); ud = np.unique(dd); pos = {d: np.where(dd == d)[0] for d in ud}
    draws = [np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))]) for _ in range(B)]
    qa = (1 - level) / 2; boot = {}
    for k in cand_sets:
        t_, p_ = cw_test(y, fb, ff[k])
        dr = np.array([(np.sum((y[i] - fb[i]) ** 2) - np.sum((y[i] - ff[k][i]) ** 2)) / np.sum((y[i] - ybar) ** 2) for i in draws])
        da = np.array([auc_(tail[i], ff[k][i]) - auc_(tail[i], fb[i]) for i in draws])
        boot[k] = dr
        out["cands"][k] = {"delta_r2": float((np.sum((y - fb) ** 2) - np.sum((y - ff[k]) ** 2)) / sst),
                           "delta_r2_ci": [float(np.quantile(dr, qa)), float(np.quantile(dr, 1 - qa))],
                           "delta_auc": float(auc_(tail, ff[k]) - auc_(tail, fb)),
                           "delta_auc_ci": [float(np.nanquantile(da, qa)), float(np.nanquantile(da, 1 - qa))],
                           "clark_west_t": t_, "clark_west_p": p_}
    out["_boot"] = boot
    return out

def paired_diff(ev, a, b, level=0.95):
    d = ev["_boot"][a] - ev["_boot"][b]; qa = (1 - level) / 2
    return {"diff": ev["cands"][a]["delta_r2"] - ev["cands"][b]["delta_r2"],
            "ci": [float(np.quantile(d, qa)), float(np.quantile(d, 1 - qa))], "p_one_sided": float(np.mean(d <= 0))}

def holm(pd_):
    items = sorted((v, k) for k, v in pd_.items()); out, run = {}, 0.0
    for j, (p_, k) in enumerate(items):
        run = max(run, min(1.0, p_ * (len(items) - j))); out[k] = run
    return out


## Universe and pre-registration (before any trade is downloaded)

In [ ]:
# ============================================================================
# CELL C2 — UNIVERSE (daily candles only; no intraday data). About 3-5 minutes; after a restart it reloads instantly.
# All USD spot pairs listed and trading on Coinbase, excluding stablecoins and wrapped/staked tokens, that already
# traded in December 2023. Liquidity from June 2024 daily candles (a training-period month):
#   thin = median daily dollar volume $0.5M-$5M;  mid control = $15M-$50M.
# Up to 20 thin and 4 mid pairs are drawn with a fixed seed. No trade data have been downloaded at this point.
# ============================================================================
def candles_daily(pid, start, end):
    c = api(f"/products/{pid}/candles", {"granularity": 86400, "start": start, "end": end})
    if not c:
        return pd.DataFrame(columns=["time", "low", "high", "open", "close", "volume"])
    return pd.DataFrame(c, columns=["time", "low", "high", "open", "close", "volume"])

_u = os.path.join(RESULTS, "C2_universe.json")
if os.path.exists(_u):                                  # after a restart: reuse the universe fixed earlier
    _m = json.load(open(_u))["metrics"]
    THIN17, MID17, OLDER["param"] = _m["thin"], _m["mid"], _m["pagination_param"]
    print("reloaded the universe fixed earlier")
else:
    prods = api("/products")
    cands = sorted(p["id"] for p in prods if p.get("quote_currency") == "USD" and p.get("status") == "online"
                   and not p.get("trading_disabled", False) and p.get("base_currency") not in STABLE)
    print(len(cands), "USD pairs")
    LIQ = {}
    for i, pid in enumerate(cands):
        dec = candles_daily(pid, "2023-12-01T00:00:00Z", "2023-12-31T00:00:00Z")
        jun = candles_daily(pid, "2024-06-01T00:00:00Z", "2024-06-30T00:00:00Z")
        if len(dec) >= 20 and len(jun) >= 25:
            LIQ[pid] = float(np.median(jun["volume"].astype(float) * jun["close"].astype(float)) / 1e6)
        if i % 50 == 0:
            print(f"   {i}/{len(cands)}")
    THIN_POOL = sorted(s for s, v in LIQ.items() if 0.5 <= v <= 5.0)
    MID_POOL = sorted(s for s, v in LIQ.items() if 15.0 <= v <= 50.0)
    R = rng(17)
    THIN17 = sorted(R.choice(THIN_POOL, min(20, len(THIN_POOL)), replace=False).tolist())
    MID17 = sorted(R.choice(MID_POOL, min(4, len(MID_POOL)), replace=False).tolist())
    check_pagination(THIN17[0])
    record("C2_universe", None, "audit", {"n_usd_pairs": len(cands), "eligible": len(LIQ), "liquidity_musd_june2024": LIQ,
           "thin_pool": THIN_POOL, "mid_pool": MID_POOL, "thin": THIN17, "mid": MID17, "pagination_param": OLDER["param"]})
print("pagination: older trades via", OLDER["param"])
print("thin:", THIN17); print("mid:", MID17)


In [ ]:
# ============================================================================
# CELL C3 — PRE-REGISTRATION (run BEFORE C4 downloads any trade). E-mail P17_new_venue.json to yourself.
# ============================================================================
PREREG_P17 = {
    "question": "On a venue never used in Parts 1-16 (Coinbase Exchange spot, USD pairs): (1) does the depth-free amplification "
                "margin C3_flow forecast the next hour's maximum adverse excursion beyond a volatility and price-impact baseline "
                "in thin pairs? (2) Is its content the minute-level alignment of flow and prices that the model describes, rather "
                "than the persistence of order flow (order splitting)? Test: the real statistic beats an order-splitting placebo.",
    "data": "Coinbase Exchange public REST trades (no account); minute bars of last trade price, signed taker notional (taker buy "
            "when the maker side is 'sell') and notional volume; a day is valid if >= 60% of its minutes have a trade; days with "
            "more than 400 pages of trades are dropped",
    "universe": {"thin": THIN17, "mid_control": MID17, "rule": "USD pairs online in 2026, excluding stablecoins and wrapped or "
                 "staked tokens, with >= 20 daily candles in Dec 2023 and >= 25 in Jun 2024; thin = median Jun-2024 daily dollar "
                 "volume $0.5M-$5M (up to 20 drawn, seed 20261134); mid control = $15M-$50M (up to 4 drawn)"},
    "days": {"train": TRAIN_DAYS, "test": TEST_DAYS},
    "inclusion": "a pair enters if it has >= 15 valid training days and >= 25 valid test days",
    "unit": "pair x hour t in 06-23 UTC; features from minutes [t-360, t); outcome MAE over (t, t+60] in basis points",
    "flow_scale": "each pair's median |flow| over training minutes with at least one trade",
    "candidate": "C3_flow = kappa^2 eta q - (1-a) kappa, from dp = kappa u - (1-a) x_lag and u = (1-eta) u_lag + eta q x_lag, "
                 "x = 1e4 log price minus EMA(60 minutes), u = signed flow / flow scale (Part 7 and Part 14 definition)",
    "placebo": "C3_placebo = the same statistic with the flow series of the window circularly shifted by 180 minutes inside the "
               "window (prices unchanged): it keeps the window's flow distribution and, apart from one wrap point, its "
               "autocorrelation (order splitting), and breaks the minute alignment between flow and price; it uses no data from "
               "outside the window",
    "baseline": CB_CTRL + ["symbol fixed effects"],
    "primary_1": "thin: MAE_60 | C3_flow beyond the baseline; Clark-West one-sided p and 95% date-block bootstrap CI of dR2",
    "primary_2": "thin: dR2(C3_flow) - dR2(C3_placebo), each beyond the same baseline on the same rows; paired 95% date-block "
                 "bootstrap CI; p = share of bootstrap draws <= 0",
    "success": "fixed-sequence testing, which controls the familywise error at 0.05: primary 1 passes if its p is below 0.05 "
               "and its 95% interval is above zero; primary 2 is tested only if primary 1 passes, and passes if its p is below "
               "0.05 and its 95% interval is above zero (beating a placebo is informative only if the real statistic has content)",
    "secondary": ["thin: S60 beyond the baseline", "thin: C3_placebo beyond the baseline (predicted: no gain)",
                  "thin: VPIN beyond the baseline", "thin: C3_flow beyond the baseline plus VPIN",
                  "mid control: C3_flow beyond the baseline"],
    "inference": "all thin-pair candidates are evaluated on the same rows (hours where C3_flow, C3_placebo, S60 and VPIN all exist) and the same 2,000 date-block bootstrap draws; Clark-West with 6 Newey-West lags; secondary Holm-adjusted",
    "runs": "C6 is run once; every run is counted in prereg/test_set_unlocks_p17.json",
    "not_tested": "no guard test on this venue (a certified operator needs three extra days per evaluation day)",
}
spec17 = json.dumps(PREREG_P17, indent=2, sort_keys=True); P17_HASH = hashlib.sha256(spec17.encode()).hexdigest()
p17_path = os.path.join(PREREG, "P17_new_venue.json")
if not os.path.exists(p17_path):
    open(p17_path, "w").write(spec17); print("pre-registration written:", p17_path)
    if IN_COLAB:
        from google.colab import files
        files.download(p17_path)
elif hashlib.sha256(open(p17_path).read().encode()).hexdigest() != P17_HASH:
    raise RuntimeError("The spec differs from the registered file. Do not edit the spec after registering it.")
print("spec SHA-256:", P17_HASH)


## Data and hourly blocks

In [ ]:
# ============================================================================
# CELL C4 — DATA (trades -> minute bars). Cached in Drive per pair-day; SAFE TO RERUN after a disconnect: it resumes.
# Expect roughly 2-3 hours at Coinbase's public rate limit. Progress is printed every 50 pair-days.
# ============================================================================
assert os.path.exists(p17_path) and hashlib.sha256(open(p17_path).read().encode()).hexdigest() == P17_HASH, "run C3 first"
jobs = [(s, d) for s in THIN17 + MID17 for d in TRAIN_DAYS + TEST_DAYS]
PANEL, VST = {}, []
t_start = time.time()
with ThreadPoolExecutor(4) as ex:                       # the shared rate limiter keeps the total at 7 requests/s
    futs = {ex.submit(get_day, s, d): (s, d) for s, d in jobs}
    for i, f in enumerate(as_completed(futs), 1):
        s, d = futs[f]
        try:
            df, st = f.result(); VST.append(st)
            if df is not None and st.get("day_ok"):
                PANEL[(s, d)] = df
        except Exception as e:
            VST.append({"symbol": s, "date": d, "reason": f"error: {e}"})
        if i % 50 == 0:
            el = (time.time() - t_start) / 60
            print(f"   {i}/{len(jobs)} pair-days, {el:.0f} min, about {el / i * (len(jobs) - i):.0f} min left, {N_REQ[0]} requests")
V = pd.DataFrame(VST)
FLOW_SCALE17 = {}
for s in THIN17 + MID17:
    tr = [PANEL[(s, d)] for d in TRAIN_DAYS if (s, d) in PANEL]
    v = np.concatenate([np.abs(x["flow"].to_numpy()[x["ntr"].to_numpy() > 0]) for x in tr]) if tr else np.array([])
    FLOW_SCALE17[s] = float(np.median(v)) if len(v) else np.nan
ntr = {s: sum((s, d) in PANEL for d in TRAIN_DAYS) for s in THIN17 + MID17}
nte = {s: sum((s, d) in PANEL for d in TEST_DAYS) for s in THIN17 + MID17}
IN_THIN = [s for s in THIN17 if ntr[s] >= 15 and nte[s] >= 25 and np.isfinite(FLOW_SCALE17[s]) and FLOW_SCALE17[s] > 0]
IN_MID = [s for s in MID17 if ntr[s] >= 15 and nte[s] >= 25 and np.isfinite(FLOW_SCALE17[s]) and FLOW_SCALE17[s] > 0]
man = V[[c for c in ["symbol", "date", "trades", "pages", "minute_coverage", "day_ok", "first_id", "last_id", "sha256", "reason"] if c in V.columns]]
man.to_csv(os.path.join(RESULTS, "p17_manifest.csv"), index=False)
print("thin pairs entering:", IN_THIN); print("mid pairs entering:", IN_MID)
record("C4_build", None, "audit", {"requested": len(jobs), "valid": len(PANEL), "train_days": ntr, "test_days": nte,
       "flow_scale": FLOW_SCALE17, "in_thin": IN_THIN, "in_mid": IN_MID, "requests": N_REQ[0],
       "reasons": V["reason"].value_counts().to_dict() if "reason" in V.columns else {}})


In [ ]:
# ============================================================================
# CELL C5 — HOURLY BLOCKS (features, placebo and outcome; no evaluation here). About 10-20 minutes.
# ============================================================================
rows = []
for (s, d), df in sorted(PANEL.items()):
    if s in IN_THIN + IN_MID:
        rows += cb_blocks(df, s, d, FLOW_SCALE17[s])
CB = pd.DataFrame(rows)
CB["group"] = np.where(CB["symbol"].isin(IN_THIN), "thin", "mid")
CB["split"] = np.where(CB["date"] <= TRAIN_END, "train", "test")
CB["kappa_sq"] = CB["kappa_hat"] ** 2
CB = winsor_train(CB, ["kappa_hat", "kappa_sq", "kyle_lam", "amihud_60", "rv_15", "rv_day", "ewma_vol", "mae_lag60"])
CB.to_parquet(os.path.join(CDATA, "blocks_p17.parquet"))
print(CB.groupby(["group", "split"]).size().unstack())
cc = CB[["C3_flow", "C3_placebo"]].dropna()
record("C5_blocks", None, "audit", {"blocks": CB.groupby(["group", "split"]).size().unstack().to_dict(),
       "coverage": {c: float(CB[c].notna().mean()) for c in ["C3_flow", "C3_placebo", "S60", "vpin", "MAE_60"]},
       "corr_real_placebo_training": float(cc.corr(method="spearman").iloc[0, 1]) if len(cc) > 10 else None})


## The test: run ONCE

In [ ]:
# ============================================================================
# CELL C6 — THE TEST (RUN ONCE; every run is counted)
# ============================================================================
assert hashlib.sha256(open(p17_path).read().encode()).hexdigest() == P17_HASH, "spec changed after registration"
pth17 = os.path.join(PREREG, "test_set_unlocks_p17.json")
n17 = (json.load(open(pth17))["n"] if os.path.exists(pth17) else 0) + 1
json.dump({"n": n17, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": P17_HASH}, open(pth17, "w"))
TH, MD = CB[CB["group"] == "thin"], CB[CB["group"] == "mid"]
# all thin-pair candidates on the same rows (rows where real, placebo, S60 and VPIN all exist) and the same bootstrap draws
TH = TH.dropna(subset=["C3_flow", "C3_placebo", "S60", "vpin"])
ev = evaluate_set(TH, "MAE_60", {"real": ["C3_flow"], "placebo": ["C3_placebo"], "S60": ["S60"], "vpin": ["vpin"],
                                 "real_plus_vpin": ["C3_flow", "vpin"]}, CB_CTRL, seed=171)
c_real = ev["cands"]["real"]; pdiff = paired_diff(ev, "real", "placebo")
beyond_vpin = paired_diff(ev, "real_plus_vpin", "vpin")      # C3_flow's gain once VPIN is in the baseline
H = {"P1_new_venue": c_real["clark_west_p"], "P2_beats_placebo": pdiff["p_one_sided"]}     # fixed sequence: no adjustment
ok = {"P1_new_venue": bool(H["P1_new_venue"] < 0.05 and c_real["delta_r2_ci"][0] > 0)}
ok["P2_beats_placebo"] = bool(ok["P1_new_venue"] and H["P2_beats_placebo"] < 0.05 and pdiff["ci"][0] > 0)
if not ok["P1_new_venue"]:
    print("primary 1 failed, so primary 2 is not tested (fixed sequence); its statistics are reported descriptively")
ev_mid = evaluate_set(MD.dropna(subset=["C3_flow"]), "MAE_60", {"real": ["C3_flow"]}, CB_CTRL, seed=172) if len(IN_MID) else None
sec = {"S60": ev["cands"]["S60"]["clark_west_p"], "placebo": ev["cands"]["placebo"]["clark_west_p"],
       "vpin": ev["cands"]["vpin"]["clark_west_p"], "beyond_vpin": beyond_vpin["p_one_sided"]}
if ev_mid:
    sec["mid_control"] = ev_mid["cands"]["real"]["clark_west_p"]
HS = holm(sec)
for k in ok:
    print(f"=== {k}:", "PASS" if ok[k] else "FAIL", "| p =", round(H[k], 5))
print("real    dR2 = %+.3f pts %s  CW t = %.2f" % (100 * c_real["delta_r2"], np.round(100 * np.array(c_real["delta_r2_ci"]), 3), c_real["clark_west_t"]))
print("placebo dR2 = %+.3f pts %s" % (100 * ev["cands"]["placebo"]["delta_r2"], np.round(100 * np.array(ev["cands"]["placebo"]["delta_r2_ci"]), 3)))
print("real - placebo = %+.3f pts %s" % (100 * pdiff["diff"], np.round(100 * np.array(pdiff["ci"]), 3)))
print("beyond VPIN    = %+.3f pts %s" % (100 * beyond_vpin["diff"], np.round(100 * np.array(beyond_vpin["ci"]), 3)))
print("secondary Holm:", {k: round(v, 4) for k, v in HS.items()})
strip = lambda e: {k: v for k, v in e.items() if k != "_boot"} if e else None
record("C6_new_venue_test", bool(all(ok.values())), "real-data",
       {"thin": IN_THIN, "mid": IN_MID, "thin_eval": strip(ev), "real_minus_placebo": pdiff, "beyond_vpin": beyond_vpin,
        "mid_eval": strip(ev_mid), "p_primaries_fixed_sequence": H, "pass": ok, "secondary_holm_p": HS,
        "prereg_sha256": P17_HASH, "test_set_unlocks": n17}, criterion=PREREG_P17["success"])


In [ ]:
# ============================================================================
# CELL C7 — ZIP THE RESULTS (send p17_results.zip to Claude)
# ============================================================================
zp = os.path.join(BASE, "p17_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [os.path.join(RESULTS, f + ".json") for f in ("C2_universe", "C4_build", "C5_blocks", "C6_new_venue_test")] + [os.path.join(RESULTS, "p17_manifest.csv")]:
        if os.path.exists(f):
            z.write(f, "results/" + os.path.basename(f))
    for f in [p17_path, os.path.join(PREREG, "test_set_unlocks_p17.json")]:
        if os.path.exists(f):
            z.write(f, "prereg/" + os.path.basename(f))
print("wrote", zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)


## How to read C6
| Result | Meaning for Paper 2 |
|---|---|
| Both pass | The thin-book result holds on a fourth venue, and its content is the flow–price alignment, not order-flow persistence. |
| 1 passes, 2 fails | The statistic forecasts on Coinbase, but a placebo with the same flow persistence does about as well: the content may be order splitting, not feedback. |
| 1 fails | Coinbase joins Bybit as a venue where the effect is absent; the paper reports it with the same prominence. |